# Fresnel Zone - deep dive
- The method that we have been using is to calculate $\zeta_1$, and $\zeta_2$ directly with ray traces and normalization
- There is some concern that these methods aren't valid because (at least for the $\pm$ 7 paths, the source is pretty close to the caustic)

- Here, I'm going to explore solving for $\zeta_{1/2}$ using the second derivative of c

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import pickle
import numpy as np
from n23_fluctuations_ragland import tools
import xarray as xr
import scipy
import matplotlib.pyplot as plt

load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"


with open(fn, 'rb') as f:
    ray_run = pickle.load(f)

rays = ray_run['rays']
eigs = ray_run['eigs']

rays_back = ray_run['rays_back']
eigs_back = ray_run['eigs_back']

source_depth = ray_run["geo"]["source_depth"]
reciever_range = ray_run["geo"]["receiver_range"]
num_range_save = 1000

envi = ray_run["envi"]

In [ ]:
latitude = 39.192499999
Omega = 7.292115e-5  # Earth Rotation Speed [rad/s]
f = 2 * Omega * np.sin(np.radians(latitude))  # [rad/s]

zeta0 = 7.3 * np.sqrt(2)
rvs = tools.compute_n23_ray_variables(f=f, zeta0=zeta0)

In [ ]:
fig = plt.figure()

for k, ray_id in enumerate(['+7','-7','+6','-6']):

    for i in range(len(rvs[ray_id])):
        plt.plot(rvs[ray_id][i]['rs'], np.sqrt(rvs[ray_id][i]['Rf2']), c=f'C{k}', alpha=1, label=ray_id)
        break

plt.legend()

In [ ]:
fig,axes = plt.subplots(3,4, figsize=(13,6))

pair_id = 0
for k, ray_id in enumerate(['-7','+7','-6','+6']):
    ## Compute V'' along the ray
    c0 = rvs[ray_id][pair_id]['Cray'].mean()
    
    V = 0.5 * (1 - (c0**2 / envi.sound_speed**2))
    Vpp = V.differentiate('depth').differentiate('depth')
    
    rs = rvs[ray_id][pair_id]['rs']
    zs = rvs[ray_id][pair_id]['zs']
    rsx = xr.DataArray(rs, dims=["s"])
    zsx = xr.DataArray(zs, dims=["s"])
    
    Vpp_ray = Vpp.interp({"range": rsx, "depth": -zsx})
    
    Vpp_ray.name = '$\\frac{d^2 c}{dz^2}$'
    
    Vpp_ray_interp = scipy.interpolate.interp1d(rs, Vpp_ray.values)
    
    ## integrate the ODEs
    def ode(x, y):
        zeta, dzeta = y
        return [dzeta, -Vpp_ray_interp(x) * zeta]
    
    y0_1 = [0.0, 1.0]  # zeta1(0)=0, zeta1'(0)=1
    sol1 = scipy.integrate.solve_ivp(ode, (0.0, rs[-1]), y0_1, t_eval=rs, rtol=1e-9, atol=1e-12)
    
    zeta1 = sol1.y[0]
    zeta1p = sol1.y[1]
    
    y0_2 = [0.0, -1.0]
    sol2 = scipy.integrate.solve_ivp(ode, (rs[-1], 0.0), y0_2, t_eval=np.flip(rs), rtol=1e-9, atol=1e-12)
    
    # Reverse back to increasing x
    zeta2 = sol2.y[0][::-1]
    zeta2p = sol2.y[1][::-1]
       
    ## Compute Fresnel Zone
    lam = float(c0)/250
    
    W = ((zeta2*zeta1p) - (zeta1*zeta2p))
    Rf2_new = lam * zeta1 * zeta2 / W

    ## Add Plots

    plt.sca(axes[0,k])
    line1 = plt.plot(rs/1000, zeta1/zeta1[-1], label='$V''$ method')[0]
    line2 = plt.plot(rs/1000, rvs[ray_id][pair_id]['RT_zeta_1'], label='Ray shooting method')[0]
    plt.xlabel('range [km]')
    plt.ylabel('$\\zeta_1$ [m]')
    plt.grid()
    plt.title(f'$\\mathbf{{{ray_id}}}$')

    plt.sca(axes[1,k])
    plt.plot(rs/1000, zeta2/zeta2[0])
    plt.plot(rs/1000, rvs[ray_id][pair_id]['RT_zeta_2'])
    plt.xlabel('range [km]')
    plt.ylabel('$\\zeta_2$ [m]')
    plt.grid()

    plt.sca(axes[2,k])
    plt.plot(rs/1000, np.abs(Rf2_new))
    plt.plot(rs/1000, rvs[ray_id][pair_id]['Rf2'])
    plt.xlabel('range [km]')
    plt.ylabel('$R_f^2$ [m]')
    plt.grid()

# Create a single legend for the entire figure
leg = fig.legend([line1, line2], ["$V^{\\prime\\prime}$ method", "Ray shooting method"], 
           loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=True)

for line in leg.get_lines():
    line.set_linewidth(3)
    
plt.tight_layout(rect=[0, 0, 0.95, 1], pad=0)